In [ ]:
%pip install --quiet --upgrade langchain-text-splitters langchain-community langgraph

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.9/154.9 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 kB 7.9 MB/s eta 0:00:00


In [ ]:
!pip install -qU "langchain[openai]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.4/63.4 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.3/438.3 kB 10.4 MB/s eta 0:00:00


In [ ]:
!pip install -qU langchain-openai

In [ ]:
!pip install -qU langchain-community

In [ ]:
!pip install -qU langchain-core

In [ ]:
#export LANGSMITH_TRACING="true"
#export LANGSMITH_API_KEY="..."
import getpass
import os

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = getpass.getpass()

··········


In [ ]:
import getpass
import os

#if not os.environ.get("OPENAI_API_KEY"):
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain.chat_models import init_chat_model

#llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)
llm = init_chat_model("openai:o3-mini")

Enter API key for OpenAI: ··········


In [ ]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [ ]:
import bs4
from langchain import hub
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict

# Load and chunk contents of the blog
loader = WebBaseLoader(
    #add here a path to your GitHub repo with a folder having each of the skills in a separate txt file
    web_paths=("https://github.com/xxxx",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)

# Index chunks
_ = vector_store.add_documents(documents=all_splits)

# Define prompt for question-answering
prompt = hub.pull("rlm/rag-prompt")


# Define state for application
class State(TypedDict):
    question: str
    context: List[Document]
    answer: str


# Define application steps
def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state["question"])
    return {"context": retrieved_docs}


def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    response = llm.invoke(messages)
    return {"answer": response.content}


# Compile application and test
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

In [ ]:
# Example: SW/System Engineer
# Do the same of other roles copying the role description to the end of the line (or create a script to read them all

response = graph.invoke({"question": "You are a SE technical manager with experience with hiring entry level candidates for jobs, which of the skills – adaptable, proactive, self-directed, dependable, passionate, collaborative, communicative, problem-solver, courageous, self-aware, self-confident, willing to learn, process-oriented mindset, cloud computing, DevOps, operational systems, version control systems, open-source software - does the following job posting need? 0T1021 - Systems/Software Engineer I  (TCP_01) Job Family:  Engineering - SW Engineering (Systems). Job Family Definition: Designs, develops, troubleshoots and debugs software programs for software enhancements and new products. Develops software including operating systems, compilers, routers, networks, utilities, databases and Internet-related tools. Determines hardware compatibility and/or influences hardware design. Job Level Definition: Contributes to assignments of limited scope by applying technical concepts and theoretical knowledge acquired through specialized training, education, or previous experience. Acts as team member by providing information, analysis and recommendations in support of team efforts. Exercises independent judgment within defined parameters. Responsibilities: Codes and programs enhancements, updates, and changes for portions and subsystems of systems software, including operating systems, compliers, networking, utilities, databases, and Internet-related tools Executes established test plans and protocols for assigned portions of code; identifies, logs, and debugs assigned issues. Develops understanding of and relationship with internal and outsourced development partners on software systems design and development. Participates as a member of project team of other software systems engineers and internal and outsourced development partners to develop reliable, cost effective and high quality solutions for low to moderately- complex products. Education and Experience Required: Bachelor's or Master's degree in Computer Science, Information Systems, or equivalent Typically 0-2 years experience Knowledge and Skills: Experience or understanding of software systems design tools and languages. Good analytical and problem solving skills. Understanding of design for software systems running on multiple platform types Understanding of basic testing, coding, and debugging procedures. Good written and verbal communication skills; mastery in English and local language."})
print(response["answer"])

In [ ]:
Cloud_Dev = "0T3911 - Cloud Developer I  (TCP_01) Job Family:  Engineering - Cloud Engineering. Job Family Definition: The Cloud Developer builds from the ground up to meet the needs of mission-critical applications, and is always looking for innovative approaches to deliver end-to-end technical solutions to solve customer problems. Brings technical thinking to break down complex data and to engineer new ideas and methods for solving, prototyping, designing, and implementing cloud-based solutions. Collaborates with project managers and development partners to ensure effective and efficient delivery, deployment, operation, monitoring, and support of Cloud engagements. The Cloud Developer provides business value expertise to drive the development of innovative service offerings that enrich HPE\'s Cloud Services portfolio across multiple systems, platforms, and applications. Job Level Definition: Contributes to assignments of limited scope by applying technical concepts and theoretical knowledge acquired through specialized training, education, or previous experience. Acts as team member by providing information, analysis and recommendations in support of team efforts. Exercises independent judgment within defined parameters. Responsibilities: Develops and maintains cloud application modules per feature specifications, adhering to security policies. Designs test plans and executes and automates test cases for assigned portions of the application. Deploys code and debugs issues. Shares and reviews innovative technical ideas with peers, high-level technical contributors, technical writers, and managers. Analyses science, engineering, business, and other data processing problems to develop and implement solutions to complex application problems, system administration issues, or network concerns. Education and Experience Required: Bachelor's degree in computer science, engineering, information systems, or closely related quantitative discipline. Master\'s desirable. Typically, 0-2 years\' experience. Knowledge and Skills: Programming skills in Python, Java, Golang, or JavaScript. Understanding of basic testing, coding, and debugging procedures. Ability to quickly learn new skills and technologies and work well with other team members. Good written and verbal communication skills. Understanding DevOps practices like continuous integration/continuous deployment (CI/CD)."
Prod_Manager = "0T2311 - Product Manager I  (TCP_01) Job Family:  Engineering - Product Management. Job Family Definition: Designs, plans, develops and manages a product or portfolio of products throughout the solution portfolio lifecycle: from new product definition or enhancements to existing products; planning, design, forecasting, and production; to end of life. Job Level Definition: Contributes to assignments of limited scope by applying technical concepts and theoretical knowledge acquired through specialized training, education, or previous experience. Acts as team member by providing information, analysis and recommendations in support of team efforts. Exercises independent judgment within defined parameters. Responsibilities: Contributes to standard product development plan. Contributes towards data collation on customer requirements, target customer segments and business case to bring innovative and disruptive products to market. Collaborates closely with key stakeholders on one or more product strategy and strategy execution across all phases of the lifecycle (e.g., planning, development, launch, management, exit). Operationalizes financial targets to meet performance objectives. Education and Experience Required: Bachelor's degree or equivalent in computer science, engineering or related field of study. MBA or advanced degree in computer science or engineering preferred. 1+ years of work experience in related field. Technical understanding and knowledge of the relevant industry. Knowledge and Skills: Basic understanding of product development. Basic skills in cost modeling efficient solutions, and financial performance metric analysis. Basic business acumen and knowledge of root cause analysis and problem detection. Technical understanding and knowledge of the relevant industry and ability to provide product specific technical training to the team."
SWE_Firmware = "0T1341 - SW Engineer Firmware I  (TCP_01) Job Family:  Engineering - SW Engineering (Firmware). Job Family Definition: Analyzes, designs, programs, debugs and modifies firmware (e.g., DSP, embedded code, BIOS). Work often involves analog and digital hardware and real-time operating systems. Position requires knowledge and exposure to hardware design. Typically programs in machine language, assembly language and high-level languages (e.g., C, C++). Job Level Definition: Contributes to assignments of limited scope by applying technical concepts and theoretical knowledge acquired through specialized training, education, or previous experience. Acts as team member by providing information, analysis and recommendations in support of team efforts. Exercises independent judgment within defined parameters. Responsibilities: Codes and programs enhancements, updates, and changes for portions and subsystems of firmware, including DSP, embedded code, EFI drivers, EFI applications and BIOS/UEFI. Executes established test plans and protocols for assigned portions of code; identifies, logs, and debugs assigned issues. Develops understanding of and relationship with internal and outsourced development partners on firmware design and development. Participates as a member of project team of other firmware engineers and internal and outsourced development partners to develop reliable, cost effective and high quality solutions for low to moderately-complex products. Education and Experience Required: Bachelor's or Master's degree in Computer Science, Information Systems, Electrical Engineering, or equivalent. Typically 0-2 years experience. Knowledge and Skills: Experience or understanding of firmware design tools and languages. Good analytical and problem solving skills. Understanding of firmware and hardware design principles. Understanding of basic testing, coding, and debugging procedures. Good written and verbal communication skills; mastery in English and local language."
SWE_QA = "0T1351 - SW Engineer Quality Assurance I  (TCP_01) Job Family:  Engineering - SW Engineering (Quality Assurance). Job Family Definition: Set and maintain quality standards for company products through the use of systematic processes. Develops, modifies, and executes software test strategies, plans and suites. Analyzes and writes test standards and procedures. Maintains documentation of test results to assist in debugging and modification of software. Analyzes test results to ensure existing functionality and recommends corrective action. May develop tools and environments to automate test execution. Consults with development engineers in resolution of problems. Job Level Definition: Contributes to assignments of limited scope by applying technical concepts and theoretical knowledge acquired through specialized training, education, or previous experience. Acts as team member by providing information, analysis and recommendations in support of team efforts. Exercises independent judgment within defined parameters. Responsibilities: Executes established test plans and protocols for assigned portions of code for end-user applications, systems software, and firmware running on hardware, local, networked, and Internet- based platforms; identifies, logs, and debugs assigned issues. Codes and programs test scripts, automation, and integration activities based on specific test requirements. Develops understanding of and relationship with internal and outsourced development partners on software applications design and development. Participates as a member of project team of other software quality assurance engineers and internal and outsourced quality assurance partners to develop reliable, cost effective and high quality solutions for low to moderately-complex products. Education and Experience Required: Bachelor's or Master's degree in Computer Science, Information Systems, or equivalent. Typically 0-2 years experience. Knowledge and Skills: Experience or understanding of software quality assurance tools and processes. Understanding of basic testing, coding, and debugging procedures. Good analytical and problem solving skills. Basic understanding of design for software and firmware running on multiple platform types Good written and verbal communication skills; mastery in English and local language."
SWD = "0T1301 - Software Designer I  (TCP_01) Job Family:  Engineering - SW Engr (Applications). Job Family Definition: Analyzes, designs, programs, debugs, and modifies software enhancements and/or new products used in local, networked, or Internet- related computer programs, primarily for end users. Using current programming language and technologies, writes code, completes programming, and performs testing and debugging of applications. Completes documentation and procedures for installation and maintenance. May interact with users to define system requirements and/or necessary modifications. Management Level Definition: Contributes to assignments of limited scope by applying technical concepts and theoretical knowledge acquired through specialized training, education, or previous experience. Acts as team member by providing information, analysis and recommendations in support of team efforts. Exercises independent judgment within defined parameters. Responsibilities: Codes and programs enhancements, updates, and changes for portions and subsystems of end-user applications software running on local, networked, and Internet-based platforms based on specific requirements and instructions. Executes established test plans and protocols for assigned portions of code; identifies, logs, and debugs assigned issues. Develops understanding of and relationship with internal and outsourced development partners on software applications design and development. Participates as a member of project team of other software applications engineers and internal and outsourced development partners to develop reliable, cost effective and high quality solutions for low to moderately- complex products. Education and Experience Required: Bachelor's or Master's degree in Computer Science, Information Systems, or equivalent. Typically 0-2 years experience. Knowledge and Skills: Experience or understanding of software applications design tools and languages. Good analytical and problem solving skills. Understanding of design for software applications running on multiple platform types. Understanding of basic testing, coding, and debugging procedures. Good written and verbal communication skills; mastery in English and local language."
Sys_SWE = "0T1021 - Systems/Software Engineer I  (TCP_01) Job Family:  Engineering - SW Engineering (Systems). Job Family Definition: Designs, develops, troubleshoots and debugs software programs for software enhancements and new products. Develops software including operating systems, compilers, routers, networks, utilities, databases and Internet-related tools. Determines hardware compatibility and/or influences hardware design. Job Level Definition: Contributes to assignments of limited scope by applying technical concepts and theoretical knowledge acquired through specialized training, education, or previous experience. Acts as team member by providing information, analysis and recommendations in support of team efforts. Exercises independent judgment within defined parameters. Responsibilities: Codes and programs enhancements, updates, and changes for portions and subsystems of systems software, including operating systems, compliers, networking, utilities, databases, and Internet-related tools Executes established test plans and protocols for assigned portions of code; identifies, logs, and debugs assigned issues. Develops understanding of and relationship with internal and outsourced development partners on software systems design and development. Participates as a member of project team of other software systems engineers and internal and outsourced development partners to develop reliable, cost effective and high quality solutions for low to moderately- complex products. Education and Experience Required: Bachelor's or Master's degree in Computer Science, Information Systems, or equivalent Typically 0-2 years experience Knowledge and Skills: Experience or understanding of software systems design tools and languages. Good analytical and problem solving skills. Understanding of design for software systems running on multiple platform types Understanding of basic testing, coding, and debugging procedures. Good written and verbal communication skills; mastery in English and local language."
Test_Engineer = "0T1041 - Test Engineer I  (TCP_01) Job Family:  Engineering - Test Engineering. Job Family Definition: Responsible for planning and arranging the labor, schedules, equipment and diagnostics required for testing and evaluating both standard and special devices. Specifies tests to be performed and provides test area with parameters for sample testing. Compiles data and defines changes required in testing equipment and diagnostics, testing procedures, manufacturing processes, or new testing requirements. Responsible for designing, developing and implementing cost-effective methods of testing and troubleshooting systems and equipment. Job Level Definition: Contributes to assignments of limited scope by applying technical concepts and theoretical knowledge acquired through specialized training, education, or previous experience. Acts as team member by providing information, analysis and recommendations in support of team efforts. Exercises independent judgment within defined parameters. Responsibilities: Designs portions of engineering solutions to test and evaluate systems, equipment, and devices based on established engineering principles and in accordance with provided specifications and requirements. Implements and executes established test plans, schedules, and requirements for subsystems of existing designs; builds testing tooling, fixtures, and apparatuses based on provided specifications. Develops understanding of and relationship with internal and outsourced partners for testing and development. Participates as a member of project team of other test engineers and internal and outsourced testing partners to develop and execute reliable, cost effective and high quality test solutions for low to moderately-complex products. Education and Experience Required: Bachelor's or Master's degree in Computer Science, Information Systems, Electrical Engineering, or equivalent. Typically 0-2 years experience. Knowledge and Skills: Experience or understanding of testing tools and software packages. Good analytical and problem solving skills. Basic understanding of material properties and hardware and electrical component design Good written and verbal communication skills; mastery in English and local language."

In [ ]:
jobs = [Cloud_Dev,Prod_Manager,SWE_Firmware,SWE_QA,SWD, Sys_SWE, Test_Engineer]
question = "For every skill you detect in the job posting, provide: the exact excerpt or sentence from the job posting where the skill appears, and the specific wording or expression from the skill description that you used to trigger your classification of that skill, and specify the skills within the list adaptable, cloud computing, collaborative, courageous, dependable, devops, open source software, operational systems, passionate, proactive, problem solver, process-oriented mindset, self-aware, self-confident, self-directive, version control systems, and willingness to learn. Format your response as a list of skill–evidence pairs. Here is the job posting: "
responses = []
all_prompt = []

In [ ]:
for job in jobs:
  single_prompt = question + job
  response = graph.invoke({"question": single_prompt})
  responses.append(response["answer"])
  all_prompt.append(single_prompt)


In [ ]:
for index in range(len(all_prompt)):
  print(all_prompt[index])
  print(responses[index])
  print()